In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Recurrent Neural Network & Perceptron Model in Psyre

This notebook demonstrates the unrolled RNN + ReLU perceptron model from
`examples/src/rnn.py` (`DIMENSION=5`), running identically under `SYNC` and
`KAHN` semantics during inference and rendering interactive Stage-1 `SYNC`
schematics. Bidirectional reverse-mode BPTT uses bounded reverse-time lookahead
(`post`) following [Potop-Butucaru, Cohen, Plotkin & Pompougnac
(2023, arXiv:2311.16977)](https://arxiv.org/abs/2311.16977), while top-level
`INFERENCE` / `TRAINING` mode control and nested initialization/gradient-descent
machines follow hierarchical synchronous mode automata
([Colaço, Pagano & Pouzet, EMSOFT '05](https://doi.org/10.1145/1086228.1086261)).

In [ ]:
# @title Psyre imports

import logging
import importlib.util as _importlib_util
import os as _os
import pathlib as _pathlib
import shutil as _shutil
import subprocess as _subprocess
import sys as _sys

_repo_root = next(
    (
        _p
        for _p in (
            _pathlib.Path.cwd(),
            _pathlib.Path.cwd().parent,
            _pathlib.Path.cwd() / 'psyre',
        )
        if (_p / 'src' / 'psyre').is_dir() and (_p / 'examples' / 'src').is_dir()
    ),
    None,
)
if _repo_root is None:
  _repo_root = _pathlib.Path.cwd() / 'psyre'
  _subprocess.run(
      [
          'git',
          'clone',
          '--depth=1',
          'https://github.com/google-deepmind/psyre.git',
          str(_repo_root),
      ],
      check=True,
  )
if _importlib_util.find_spec('pydot') is None:
  _subprocess.run(
      [_sys.executable, '-m', 'pip', 'install', '-q', 'pydot>=1.4.2'],
      check=True,
  )
for _path in (_repo_root, _repo_root / 'src'):
  if str(_path) not in _sys.path:
    _sys.path.insert(0, str(_path))
if (
    _shutil.which('dot') is None
    and getattr(_os, 'geteuid', lambda: -1)() == 0
    and _shutil.which('apt-get') is not None
):
  _subprocess.run(['apt-get', '-qq', 'update'], check=True)
  _subprocess.run(['apt-get', '-qq', 'install', '-y', 'graphviz'], check=True)

# pylint: disable=g-importing-member,g-multiple-import
from psyre.builder import build_graph
from psyre.common import (
    Semantics,
    logger,
)
from examples.src.rnn import (
    DEFAULT_MAX_STEPS,
    DIMENSION,
    HyperParams,
    Mode,
    ModelParams,
    PerceptronParams,
    RnnParams,
    TrainingState,
    extract_params,
    forward,
    hierarchical_model,
)
from psyre.scheduler import Scheduler
from psyre.stream import Stream
from psyre.trace import (
    is_present,
    present_values,
)
from psyre.visualize import visualize_graph

# pylint: enable=g-importing-member,g-multiple-import

In [ ]:
# @title Psyre model inference

logger.setLevel(logging.WARNING)
samples = [
    ([1.0, 2.0, -1.0, 0.0, 1.0], 2.0),
    ([0.5, -0.5, 1.0, 0.5, -1.0], 1.5),
]
params = ModelParams(
    rnn=RnnParams(wxh=0.2, whh=0.1, bh=0.0),
    per=PerceptronParams(wxh=[0.3] * DIMENSION, bh=[0.1] * DIMENSION),
)

horizon = (1 + len(samples)) * DIMENSION
_, _, _, eval_loss = Scheduler.run(
    hierarchical_model,
    dataset=samples,
    params=params,
    initial_mode=Mode.INFERENCE,
    horizon=horizon,
)
g_infer = Scheduler.graph
present_ticks = [idx for idx in range(horizon) if is_present(eval_loss, idx)]
eval_losses = present_values(eval_loss)
_, _, _, kahn_eval_loss = Scheduler.run(
    hierarchical_model,
    dataset=samples,
    params=params,
    initial_mode=Mode.INFERENCE,
    horizon=horizon,
    semantics=Semantics.KAHN,
)
assert not Scheduler.deadlocked
assert kahn_eval_loss == eval_losses
print('Present evaluation ticks:', present_ticks)
print('Evaluation losses (SYNC == KAHN):', eval_losses)

In [ ]:
# @title Psyre model training (DIMENSION=5, DEFAULT_MAX_STEPS=3)

train_horizon = (1 + DEFAULT_MAX_STEPS) * DIMENSION
_, param_traces, train_loss, _ = Scheduler.run(
    hierarchical_model,
    dataset=samples,
    initial_mode=Mode.TRAINING,
    hyperparams=HyperParams(max_steps=DEFAULT_MAX_STEPS, holdout_index=0),
    horizon=train_horizon,
)
g_train = Scheduler.graph
train_ticks = [
    idx for idx in range(train_horizon) if is_present(train_loss, idx)
]
train_losses = present_values(train_loss)
trained_params = extract_params(param_traces)
print('Present training ticks:', train_ticks)
print('Training losses:', train_losses)
print('Trained RNN params:', trained_params.rnn)

In [ ]:
# @title Stage-1 schematic: 2-layer RNN + perceptron forward pass (DIMENSION=5)


def forward_circuit() -> Stream:
  """Constructs the 2-layer RNN + ReLU perceptron forward circuit."""
  x = Stream(samples[0][0], name='x')
  param_streams = ModelParams(
      rnn=RnnParams(
          wxh=Stream(params.rnn.wxh, name='rnn.wxh'),
          whh=Stream(params.rnn.whh, name='rnn.whh'),
          bh=Stream(params.rnn.bh, name='rnn.bh'),
      ),
      per=PerceptronParams(
          wxh=Stream(params.per.wxh, name='per.wxh'),
          bh=Stream(params.per.bh, name='per.bh'),
      ),
  )
  return forward(x, param_streams)


g_forward = build_graph(forward_circuit)
visualize_graph(
    g_forward,
    title='Stage-1 SYNC Forward Pass (DIMENSION=5)',
    height_px=560,
)

In [ ]:
# @title Stage-1 schematic: hierarchical inference machine (DIMENSION=5)

visualize_graph(
    g_infer,
    title='Stage-1 SYNC Hierarchical Model (Mode.INFERENCE, DIMENSION=5)',
    height_px=640,
)

In [ ]:
# @title Stage-1 schematic: full training mode (DIMENSION=5, DEFAULT_MAX_STEPS=3)

visualize_graph(
    g_train,
    title=(
        'Stage-1 SYNC Hierarchical Model '
        '(Mode.TRAINING, DIMENSION=5, DEFAULT_MAX_STEPS=3)'
    ),
    height_px=680,
)

In [ ]:
# @title Stage-1 schematic: full hierarchical model (training + inference)


def full_hierarchical_circuit():
  """Constructs the two-mode training and inference hierarchical machine."""
  switch = Stream.once((1 + len(samples)) * DIMENSION - 1, name='switch_mode')
  return hierarchical_model(
      dataset=samples,
      params=params,
      initial_mode=Mode.INFERENCE,
      switch_mode=switch,
      train_init_state=TrainingState.INIT_LOAD,
      hyperparams=HyperParams(max_steps=DEFAULT_MAX_STEPS),
  )


g_hierarchical = build_graph(full_hierarchical_circuit)
visualize_graph(
    g_hierarchical,
    title='Stage-1 SYNC Full Hierarchical Model (Inference -> Training)',
    height_px=680,
)